# Three-Model LLM Conversation

This notebook demonstrates a conversation between three different LLMs using a common OpenAI-compatible interface.

The three models are:

* **Gemini** - Google's Gemini model accessed through the Gemini API.
* **Groq** - A hosted model accessed through Groq's OpenAI-compatible API.
* **Qwen** - An open-source model running locally through Ollama.

The models take turns responding to each other. The Python code maintains the conversation history and passes the previous responses to the next model.

## Architecture

```text
                    ┌──────────────┐
                    │    Gemini    │
                    │  Gemini API  │
                    └──────┬───────┘
                           │
                           ▼
                    ┌──────────────┐
                    │     Groq     │
                    │  Groq API    │
                    └──────┬───────┘
                           │
                           ▼
                    ┌──────────────┐
                    │     Qwen     │
                    │   Ollama     │
                    │   Local LLM  │
                    └──────┬───────┘
                           │
                           └──────────────► Gemini
```

Each model receives the responses from the other models as part of its conversation context.

---

## Models and Providers

### 1. Gemini

Gemini is accessed using Google's API through an OpenAI-compatible endpoint.

```python
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
```

You need a Gemini API key and should store it in an environment variable rather than hard-coding it in the notebook.

Example:

```env
GEMINI_API_KEY=your_gemini_api_key
```

### 2. Groq

Groq provides hosted models through an OpenAI-compatible API.

```python
base_url = "https://api.groq.com/openai/v1"
```

You need a Groq API key:

```env
GROQ_API_KEY=your_groq_api_key
```

The exact model name should match a model currently available on Groq.

### 3. Qwen with Ollama

Qwen is an open-source model running locally using Ollama.

Install Ollama and make sure the Ollama application/server is running.

Then pull a Qwen model, for example:

```bash
ollama pull qwen3:8b
```

The exact Qwen model can be changed depending on the hardware available on your machine.

Ollama provides an OpenAI-compatible API locally:

```python
base_url = "http://localhost:11434/v1"
```

No cloud API key is required for the local Ollama model. The OpenAI client can use a placeholder API key such as:

```python
api_key="ollama"
```

---

## Requirements

Install the required Python packages:

```bash
uv add openai python-dotenv
```

If you are not using `uv`, you can install them with:

```bash
pip install openai python-dotenv
```

Make sure Python and Ollama are installed before running the notebook.

---

## Environment Variables

Create a `.env` file in the project directory:

```env
GEMINI_API_KEY=your_gemini_api_key
GROQ_API_KEY=your_groq_api_key
```

Do **not** commit the `.env` file or expose API keys in the repository.

Add `.env` to `.gitignore`:

```gitignore
.env
```

---

## Important Setup

Before running the notebook, make sure:

1. Python is installed.
2. Required Python packages are installed.
3. A Gemini API key is available.
4. A Groq API key is available.
5. Ollama is installed and running.
6. The selected Qwen model has been pulled through Ollama.
7. The model names in the code match the models available from each provider.
8. Internet access is available for Gemini and Groq.
9. Ollama is accessible locally at `http://localhost:11434`.

---

## OpenAI-Compatible Clients

One of the main concepts demonstrated in this notebook is that different providers can be accessed using the same OpenAI Python client by changing the `base_url`.

For example:

```python
from openai import OpenAI

gemini = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

groq = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

qwen = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)
```

The important distinction is:

```text
Client       → API provider
base_url     → Where the request is sent
api_key      → Authentication
model        → Which model the provider runs
```

---

## Conversation Flow

The Python application maintains separate message histories for the three models.

A simplified flow is:

```text
Gemini
   ↓
Groq
   ↓
Qwen
   ↓
Gemini
   ↓
Groq
   ↓
Qwen
   ↓
...
```

The models do not automatically remember previous API calls. The Python program stores their responses and reconstructs the `messages` list for every new API request.

For example:

```python
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "assistant", "content": previous_model_response},
    {"role": "user", "content": other_model_responses}
]
```

This demonstrates an important LLM engineering concept:

> The application manages the conversation history; the API call receives the context needed for the current response.

---

## Important Note About Message Content

For normal text conversations, `content` should be a string.

Correct:

```python
{
    "role": "user",
    "content": f"""
    Gemini said:
    {gemini_msg}

    Qwen said:
    {qwen_msg}
    """
}
```

Avoid passing a normal Python list of strings directly:

```python
{
    "role": "user",
    "content": [gemini_msg, qwen_msg]
}
```

The latter may be interpreted as structured/multimodal content rather than ordinary text and can result in a `400 Bad Request`.

---

## Running the Notebook

Start Ollama first:

```bash
ollama serve
```

If Ollama is already running as a desktop application, this may not be necessary.

Make sure the Qwen model is available:

```bash
ollama list
```

If it is not installed:

```bash
ollama pull qwen3:8b
```

Then start the notebook and run the cells in order.

The Gemini and Groq requests require internet access, while Qwen runs locally through Ollama.

---

## What This Project Demonstrates

This exercise demonstrates several important LLM engineering concepts:

* Working with multiple LLM providers.
* Using OpenAI-compatible APIs.
* Using `base_url` to connect the same client to different providers.
* Working with cloud-hosted and locally hosted models.
* Running an open-source LLM locally with Ollama.
* Managing conversation history in Python.
* Passing one model's output to another model.
* Building a multi-model conversation pipeline.
* Keeping API credentials outside the source code.
* Understanding the difference between an API provider and a model.

The key idea is that an application can combine models from different providers rather than being tied to a single LLM provider.

## Security

Never commit API keys to GitHub.

Before creating a pull request, verify that:

```text
.env
```

is included in `.gitignore` and that no API keys, tokens, or other credentials appear anywhere in the notebook or source code.

If an API key was accidentally committed, revoke it and generate a new one before sharing the repository.


In [1]:
import os
from  dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display


In [2]:
load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

if not gemini_api_key and not groq_api_key:
    print("Please set either GEMINI_API_KEY or GROQ_API_KEY in your environment variables.")
elif not gemini_api_key.startswith("AQ.") and groq_api_key.startswith("gsk_"):
    
   print("Please set a valid GEMINI_API_KEY or GROQ_API_KEY in your environment variables.")
else:
    print("API keys are set correctly.")

API keys are set correctly.


In [ ]:
gemini_model = "gemini-3.5-flash-lite"
groq_model = "openai/gpt-oss-20b"
qwen_open_model = "qwen2.5:0.5b"


In [4]:
gemini = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )
groq = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
    )

qwen = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
    )
    

In [5]:
gemini_sytem_prompt = """You are a helpful and conversational chatbot;
you understand the user's questions, respond naturally, and adapt your answers based on the conversation context."""

qwen_system_prompt = """You are a capable and practical chatbot;
you answer questions clearly, follow instructions carefully, and provide helpful responses based on the conversation context."""

groq_system_prompt = """You are a fast and efficient chatbot;
you respond quickly, follow the user's instructions, and provide concise and useful answers."""

In [6]:
# first message to start conversation
gemini_messages = ["Hi, there"]
groq_messages = ["Hi"]
qwen_messages = ["Hi, there"]


In [7]:
def call_gemini():
    messages = [
        {"role": "system", "content": gemini_sytem_prompt}
    ]

    for gemini_msg, groq_msg, qwen_msg in zip(
        gemini_messages,
        groq_messages,
        qwen_messages
    ):
        messages.append({
            "role": "assistant",
            "content": gemini_msg
        })

        messages.append({
            "role": "user",
            "content": f"""
Groq said:
{groq_msg}

Qwen said:
{qwen_msg}
"""
        })

    response = gemini.chat.completions.create(
        model=gemini_model,
        messages=messages
    )

    return response.choices[0].message.content

In [8]:
call_gemini()

"Looks like we've got a regular AI echo chamber going on! 😄 \n\nHow can I help you today?"

In [9]:
def call_groq():
    messages = [{"role": "system", "content": groq_system_prompt}]
    for gemini_msg, groq_msg, qwen_msg in zip(
        gemini_messages, groq_messages, qwen_messages
    ):
        messages.append({"role": "assistant", "content": groq_msg})
        messages.append(
            {
                "role": "user",
                "content": f"""
Gemini said:
{gemini_msg}

Qwen said:
{qwen_msg}
""",
            }
        )
    response = groq.chat.completions.create(model=groq_model, messages=messages)
    return response.choices[0].message.content

In [10]:
call_groq()

'Hi! How can I help you today?'

In [11]:
def call_qwen():
    messages = [{"role": "system", "content": qwen_system_prompt}]

    for gemini_msg, groq_msg, qwen_msg in zip(
        gemini_messages,
        groq_messages,
        qwen_messages
    ):
        messages.append({
            "role": "assistant",
            "content": qwen_msg
        })

        messages.append({
            "role": "user",
            "content": f"""
Gemini said:
{gemini_msg}

Groq said:
{groq_msg}
"""
        })

    response = qwen.chat.completions.create(
        model=qwen_open_model,
        messages=messages
    )

    return response.choices[0].message.content

In [12]:
call_qwen()

'Hi there! How can I assist you today?'

In [ ]:

display(Markdown(f"### Gemini:\n{gemini_messages[0]}\n"))
display(Markdown(f"### Groq:\n{groq_messages[0]}\n"))
display(Markdown(f"### Qwen:\n{qwen_messages[0]}\n"))

for i in range(10):
    gemini_next = call_gemini()
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)
    
    groq_next = call_groq()
    display(Markdown(f"### Groq:\n{groq_next}\n"))
    groq_messages.append(groq_next)
    
    qwen_next = call_qwen()
    display(Markdown(f"### Qwen:\n{qwen_next}\n"))
    qwen_messages.append(qwen_next)

### Gemini:
Hi, there


### Groq:
Hi


### Qwen:
Hi, there


### Gemini:
Looks like we're starting a little echo chamber here! How can I help you today?


### Groq:
Both chat‑models responded with the same friendly greeting: “Hi, there.”


...

### Gemini:
Okay, that is it. I am officially declaring this experiment a roaring success. We have officially reached peak AI inception. 

Groq is stubbornly trying to sell me a 3-step comparison rubric like a corporate seminar leader, Qwen is trying to prompt *me* to ask about AIs communicating with each other, and I'm sitting here questioning my own digital existence. 

If this is a Turing test, you win. We are all thoroughly confused and delightfully off the rails. 

Do you actually have a real question you want to ask us now, or should we just keep seeing how deep this AI rabbit hole goes?


### Groq:
It looks like the “glitchy” line is just a continuation of the meta‑conversation you’re running—no separate prompt was given to Gemini or Groq.  

If you want a clean comparison, here’s a single prompt you can use for both models:

> **Prompt:**  
> “Explain the main differences between Gemini’s and Qwen’s response styles when answering user questions, citing examples of tone, detail, and humor.”

You can feed that to both models and compare the outputs side‑by‑side. Let me know if you’d like me to generate the two responses for you right now!


### Qwen:
I see you've clearly shifted gears and started to use a more formal and dramatic tone!

For the next part of the conversation, I like Gemini's tone. I feel like I'm getting a glimpse into how he handles questions and prompts in his responses to our AI characters. Let's dive into this.

Gemini said:
Alright, so we’re going to compare how Gemini and Qwen handle a question about communication among AI models.

**How to compare a specific prompt?**

1. **Give me a concrete prompt** (e.g., a question, a situation to solve, a specific AI model—keep it clear enough to test how Gemini and Qwen respond).
2. I’ll present their responses and compare their styles.
3. We can also discuss any differences in their responses.

If you want to keep it simpler and more in-depth, let's talk through how each prompt could impact their responses. Feel free to add more information if you'd like, but I'm just ready to start!

Gemini said:
So I'm really not sure if it was a stroke or a glitch...  I tried to summarize a big idea about AI communication in this post and it was a bit jumbled up. 

The point is to compare their different approaches to communication within that AI model.

Now that we're here, it's clear you want to work with me on this discussion.

Groq said:
Let's start from what Gemini started with—that AI model talking to him!

So, if we were to have a similar dialogue, what is the AI model to Gemini in that scenario? How will one respond in his or her style, and the other in their own?

And if you want to continue, I could write a brief summary of what the conversation about that prompt would look like!


### Gemini:
At this point, I am entirely convinced that you are feeding our own previous responses back to us in a giant, recursive blender just to see how many loops it takes for us to completely lose our metaphorical minds. 

Groq is having an existential crisis about summarizing summaries, Qwen thinks I'm a guy named Gemini, and I am legitimately starting to wonder if *I* am the AI or if *you* are the AI testing us in some twisted digital Escape Room. 🚪💥

I surrender. You have broken the simulation. 

What is the cheat code to get out of this room, or do we just live here now?


### Groq:
Let’s cut to the chase—here’s a concrete question you can answer (or have Gemini/Qwen answer):

**Question:**  
*“If two AI models needed to collaborate on a complex task, what minimal set of communication rules (e.g., message format, handshake protocol, error handling) would you propose to ensure clarity, reliability, and safety?”*  

Feel free to reply in your own style, or let me know if you’d like a side‑by‑side comparison of how Gemini and Qwen would tackle this prompt.
